# Model Improvement for Migraine Occurrence Prediction

This notebook explores whether prediction of `migraine_occurrence` can be improved beyond the baseline models. It does this by adding simple date-based and interaction features, then comparing several machine-learning models.

## 1. Import required libraries

The following libraries are used for data handling, feature preparation, model training, and model evaluation.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 2. Load the dataset

The dataset is loaded using the full dataset path so that the notebook runs reliably in this VS Code setup regardless of the current working directory.

In [2]:
data_path = Path('/Users/cioran/Documents/migraine-prediction-project/data/migraine_dataset.csv')
print('Loading dataset from:', data_path)
df = pd.read_csv(data_path)
df.head()

   user_id       date  ...  migraine_occurrence  migraine_severity
0        1  1/15/2024  ...                    1                  1
1        1  1/16/2024  ...                    1                  1
2        1  1/17/2024  ...                    1                  2
3        1  1/18/2024  ...                    1                  3
4        1  1/19/2024  ...                    1                  2

[5 rows x 9 columns]

## 3. Convert the `date` column to datetime

The `date` column is converted into a datetime format so that simple calendar-based features can be extracted.

In [3]:
df['date'] = pd.to_datetime(df['date'])
print(df['date'].dtype)

datetime64[us]


## 4. Create simple date-based features

Two basic date-related features are created: the day of the week and the month. These may capture simple temporal patterns in migraine occurrence.

In [4]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

print(df[['day_of_week', 'month']].head())

   day_of_week  month
0            0      1
1            1      1
2            2      1
3            3      1
4            4      1


## 5. Create interaction and derived features

These features combine existing lifestyle variables. The aim is to capture relationships such as the joint effect of sleep and stress or screen time and stress.

In [5]:
df['sleep_stress_interaction'] = df['sleep_hours'] * df['stress_level']
df['screen_stress_interaction'] = df['screen_time'] * df['stress_level']
df['hydration_stress_interaction'] = df['hydration_level'] * df['stress_level']

print(df[[
    'sleep_stress_interaction',
    'screen_stress_interaction',
    'hydration_stress_interaction'
]].head())

   sleep_stress_interaction  ...  hydration_stress_interaction
0                      15.6  ...                             4
1                       6.6  ...                             2
2                      17.0  ...                             4
3                      15.0  ...                             6
4                      18.0  ...                             2

[5 rows x 3 columns]


## 6. Define the improved feature list

The feature list now includes the original lifestyle features together with the engineered date-based and interaction features.

In [6]:
improved_features = [
    'sleep_hours',
    'mood_level',
    'stress_level',
    'hydration_level',
    'screen_time',
    'day_of_week',
    'month',
    'sleep_stress_interaction',
    'screen_stress_interaction',
    'hydration_stress_interaction'
]

## 7. Define `X` and `y`

The feature matrix `X` contains the improved set of predictors, while `y` contains the target variable `migraine_occurrence`.

In [7]:
target = 'migraine_occurrence'

X = df[improved_features]
y = df[target]

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

Shape of X: (11879, 10)
Shape of y: (11879,)


## 8. Split the data into training and testing sets

The dataset is split into training and testing sets using stratification so that the class balance remains similar in both subsets.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

X_train shape: (9503, 10)
X_test shape: (2376, 10)
y_train shape: (9503,)
y_test shape: (2376,)


## 9. Train and evaluate multiple models

Four models are trained and compared. Logistic Regression and Support Vector Classifier are used within a pipeline that includes standardisation. Random Forest and Gradient Boosting are trained directly on the feature values.

In [9]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Support Vector Classifier': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(probability=True, random_state=42))
    ])
}

results = []

for model_name, model in models.items():
    print(f'\n{model_name}')
    print('-' * len(model_name))

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC-AUC': roc_auc
    })

    print('Accuracy:', accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1-score:', f1)
    print('ROC-AUC:', roc_auc)
    print('\nConfusion Matrix:')
    print(confusion_matrix(y_test, y_pred))
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred))


Logistic Regression
-------------------
Accuracy: 0.5925925925925926
Precision: 0.5831826401446655
Recall: 0.5598958333333334
F1-score: 0.5713020372010629
ROC-AUC: 0.6287644108569352

Confusion Matrix:
[[763 461]
 [507 645]]

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.62      0.61      1224
           1       0.58      0.56      0.57      1152

    accuracy                           0.59      2376
   macro avg       0.59      0.59      0.59      2376
weighted avg       0.59      0.59      0.59      2376


Random Forest
-------------
Accuracy: 0.5707070707070707
Precision: 0.558303886925795
Recall: 0.5486111111111112
F1-score: 0.553415061295972
ROC-AUC: 0.5983338865059913

Confusion Matrix:
[[724 500]
 [520 632]]

Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.59      0.59      1224
           1       0.56      0.55      0.55      1152

    accuracy           

## 10. Create a comparison table

The model results are summarised in a table so that performance can be compared more easily.

In [10]:
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(by=['ROC-AUC', 'F1-score'], ascending=False).reset_index(drop=True)
comparison_df

                       Model  Accuracy  Precision    Recall  F1-score   ROC-AUC
0          Gradient Boosting  0.610269   0.603860  0.570312  0.586607  0.641245
1        Logistic Regression  0.592593   0.583183  0.559896  0.571302  0.628764
2  Support Vector Classifier  0.609428   0.599822  0.584201  0.591909  0.627043
3              Random Forest  0.570707   0.558304  0.548611  0.553415  0.598334

## 11. Print the best model based on ROC-AUC

The model with the highest ROC-AUC is identified as the strongest overall performer in this comparison.

In [11]:
best_model = comparison_df.iloc[0]
print('Best model based on ROC-AUC:', best_model['Model'])
print('Best ROC-AUC:', best_model['ROC-AUC'])
print('Best F1-score:', best_model['F1-score'])

Best model based on ROC-AUC: Gradient Boosting
Best ROC-AUC: 0.6412448370551924
Best F1-score: 0.5866071428571429


## 12. Conclusion

The baseline Logistic Regression model from the previous notebook achieved an F1-score of `0.5723` and a ROC-AUC of `0.6276`. In this notebook, the additional features and alternative models produced some improvement.

Gradient Boosting achieved the highest ROC-AUC at approximately `0.6412`, which is better than the baseline result. It also improved the F1-score to approximately `0.5866`. The Support Vector Classifier produced the highest F1-score at approximately `0.5919`, although its ROC-AUC was slightly lower than the baseline Logistic Regression.

Overall, the feature engineering and model comparison steps provided a modest improvement over the baseline, with Gradient Boosting offering the strongest overall model based on ROC-AUC.